In [ ]:
from scipy.special import factorial
import pickle
import numpy as np
import pandas as pd
from scipy.stats import poisson

#### helpers

Based on code from `https://github.com/prabaey/SimSUM`.

In [ ]:
class DaysAtHome:
    
    def __init__(self, coeff, bias, exp=True): 
        self.bias = bias
        self.coeff = coeff
        self.exp = exp
        
    def get_mean(self, dysp, cough, pain, nasal, fever, self_employed): 
        
        dysp = 0 if dysp == "no" else 1
        cough = 0 if cough == "no" else 1
        pain = 0 if pain == "no" else 1
        nasal = 0 if nasal == "no" else 1
        self_employed = 0 if self_employed == "no" else 1
        
        low_fever = 0
        high_fever = 0
        if fever == "low": 
            low_fever = 1
        if fever == "high": 
            high_fever = 1
        
        logit = self.bias \
                + self.coeff[0]*dysp + self.coeff[1]*cough \
                + self.coeff[2]*pain + self.coeff[3]*nasal \
                + self.coeff[4]*low_fever + self.coeff[5]*high_fever \
                + self.coeff[6]*self_employed
        
        if self.exp:
            lambda_ = np.exp(logit)
        else: 
            lambda_ = logit
        
        return lambda_

#### read data

Source data can be downloaded on `https://github.com/prabaey/SimSUM`

In [ ]:
# read source data
data_dir = "./source_data/SimSUM.csv"
df = pd.read_csv(data_dir, delimiter=";", index_col=0)

#### generate treatments and outcomes

In [ ]:
# initialize potential outcome models
model_mu0 = DaysAtHome([0.64, 0.35, 0.47, 0.011, 0.81, 1.23, -0.5], 0.010)
model_mu1 = DaysAtHome([0.51, 0.42, 0.26, 0.0051, 0.24, 0.57, -0.5], 0.16)     

In [ ]:
# init new columns
df[['M0','M1','Y0','Y1','Y','cate']] = 0

In [ ]:
# get potential outcomes
df[['M0','M1']] = df.apply(lambda r: pd.Series({
    "M0": model_mu0.get_mean(r.dysp, r.cough, r.pain, r.nasal, r.fever, r.self_empl),
    "M1": model_mu1.get_mean(r.dysp, r.cough, r.pain, r.nasal, r.fever, r.self_empl),}), axis=1)

In [ ]:
# get cate and observed outcome
df = df.assign(
    Y0=lambda d: poisson.rvs(d.M0.to_numpy()),
    Y1=lambda d: poisson.rvs(d.M1.to_numpy()),
    Y =lambda d: np.where(d.antibiotics.eq("no"), d.Y0, d.Y1),
    cate=lambda d: d.M1 - d.M0)

# drop original columns
df = df.drop('days_at_home', axis=1)

#### select and process features

In [ ]:
# one hot fever and season
df = pd.get_dummies(df, columns=['fever'], drop_first=False)
df = pd.get_dummies(df, columns=['season'], drop_first=True, prefix='', prefix_sep='')

In [ ]:
%%capture
# binarize text
df = df.replace({'no': 0, 'yes': 1})

# binazire bool
bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype('int')

In [ ]:
# rename
df['TEXT'] = df['text']
df['T'] = df['antibiotics']

In [ ]:
# select variables
df = df[['dysp', 'cough', 'pain', 'nasal', 'fever_none', 'fever_low', 'fever_high', 
   'self_empl', 'asthma', 'smoking', 'COPD', 'winter','hay_fever',
   'M0', 'M1', 'Y0', 'Y1', 'cate', 'T', 'Y', 'TEXT']]

In [ ]:
# store
df.to_csv('./datasets/synsum.csv')